# Day 1 · Session 1 — How LLMs Work

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ksuresh/fdp-llm-agentic-ai/blob/main/day1-llm-foundations/01_how_llms_work.ipynb)

---

**Duration:** ~45 minutes  
**Goal:** Build an intuition for what's happening *inside* an LLM — tokens, embeddings, attention, and why models hallucinate.

> *Adapted from Ed Donner's LLM Engineering course (week1). Simplified and contextualised for faculty.*

---

## Step 0 — Install & Imports

Run this cell once. If you already ran `pip install -r requirements.txt`, most of these are already installed.

In [ ]:
# Run once — install if not already done
!pip install openai tiktoken python-dotenv -q

In [ ]:
import os
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

# Load API keys from .env file
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✅ Setup complete")

---

## Part 1 — What Is a Token?

LLMs don't read words. They read **tokens** — chunks of text that can be a word, part of a word, or a single character.

Understanding tokens is fundamental because:
- API costs are measured in tokens (input + output)
- Context windows are measured in tokens
- Token boundaries affect model behaviour

**Rule of thumb:** 1 token ≈ 0.75 English words. So 1,000 words ≈ 1,333 tokens.

In [ ]:
# Let's see exactly how text gets split into tokens
enc = tiktoken.encoding_for_model("gpt-4o")

def show_tokens(text):
    tokens = enc.encode(text)
    print(f"Text     : {repr(text)}")
    print(f"Token IDs: {tokens}")
    print(f"Decoded  : {[enc.decode([t]) for t in tokens]}")
    print(f"Count    : {len(tokens)} tokens")
    print()

show_tokens("Hello, how are you?")
show_tokens("IIT Madras is a premier engineering institute in Chennai.")
show_tokens("नमस्ते")

### 🔍 Notice:
- The Hindi word `नमस्ते` uses more tokens than the equivalent English word — non-English text is less efficient
- Common English words are often a single token; rare words get split
- Numbers and special characters tokenise differently

**Why does this matter for you?**  
If you're building an app for Indian language content, you'll use 2–3× more tokens than English — which means higher cost and shorter effective context.

In [ ]:
# Cost estimator — how much does a typical query cost?

def estimate_cost(input_text, output_words=200, model="gpt-4o-mini"):
    input_tokens = len(enc.encode(input_text))
    output_tokens = int(output_words * 1.33)  # approx
    
    # GPT-4o-mini pricing (as of mid-2025): $0.15/M input, $0.60/M output
    input_cost  = (input_tokens  / 1_000_000) * 0.15
    output_cost = (output_tokens / 1_000_000) * 0.60
    total_usd   = input_cost + output_cost
    total_inr   = total_usd * 83.5  # approx INR rate
    
    print(f"Model         : {model}")
    print(f"Input tokens  : {input_tokens}")
    print(f"Output tokens : ~{output_tokens}")
    print(f"Cost          : ${total_usd:.6f} USD  (~₹{total_inr:.4f})")
    print()

estimate_cost("Explain Newton's laws of motion to a first year engineering student.")
estimate_cost("You are a helpful teaching assistant for an engineering college. " * 10 + 
              "Explain Newton's laws of motion to a first year engineering student.")

---

## Part 2 — The Architecture in Plain English

You don't need to understand every equation. Here's the 3-minute version:

```
Your text
    ↓
[Tokeniser]  →  splits into token IDs
    ↓
[Embedding Layer]  →  converts each token to a vector (list of numbers)
    ↓
[Transformer Blocks × N]  →  each block refines meaning using:
    • Self-Attention  — "which other tokens matter for understanding this one?"
    • Feed-Forward    — "what does this combination of context mean?"
    ↓
[Output Layer]  →  probability distribution over all possible next tokens
    ↓
[Sampling]  →  pick the next token (temperature controls randomness)
    ↓
Repeat until done
```

**The key insight:** The model doesn't "know" anything — it predicts the most likely next token given everything before it. GPT-4o has seen so much text that its predictions are remarkably useful.

---

## Part 3 — Temperature: Creativity vs Consistency

In [ ]:
def ask(prompt, temperature=1.0, model="gpt-4o-mini", max_tokens=150):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

prompt = "Write the opening line of a short poem about the monsoon in Chennai."

print("=" * 60)
print("Temperature = 0.0  (deterministic, always same answer)")
print("=" * 60)
for _ in range(3):
    print(ask(prompt, temperature=0.0))
    print("---")

print("\n" + "=" * 60)
print("Temperature = 1.5  (creative, unpredictable)")
print("=" * 60)
for _ in range(3):
    print(ask(prompt, temperature=1.5))
    print("---")

### When to use what temperature?

| Task | Recommended Temperature |
|------|------------------------|
| Factual Q&A, code generation | 0.0 – 0.3 |
| Summarisation, translation | 0.3 – 0.7 |
| Creative writing, brainstorming | 0.8 – 1.2 |
| Highly diverse / experimental | 1.2 – 2.0 |

---

## Part 4 — Context Windows

In [ ]:
# Context windows — how much can the model "remember"?

context_windows = {
    "GPT-4o (OpenAI)":        128_000,
    "Claude Opus 4 (Anthropic)": 200_000,
    "Gemini 2.5 Pro (Google)":  1_000_000,
    "LLaMA 3.2 (Meta, local)":    128_000,
    "Mistral 7B (local)":           32_000,
}

avg_words_per_page = 250

print(f"{'Model':<35} {'Context (tokens)':>18} {'≈ Pages of text':>18}")
print("-" * 73)
for model, tokens in context_windows.items():
    pages = int((tokens * 0.75) / avg_words_per_page)
    print(f"{model:<35} {tokens:>18,} {pages:>18,}")

---

## Part 5 — Why Do LLMs Hallucinate?

In [ ]:
# Classic hallucination trap — ask about a very specific local fact

questions = [
    "Who is the current Vice Chancellor of Anna University?",
    "What was the exact NIRF ranking of PSG Tech in 2024?",
    "Name the faculty member who won the best paper award at IEEE INDICON 2023 from your institution.",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q, temperature=0)}")
    print("-" * 60)

### Why does this happen?

The model is a **next-token predictor**. When it doesn't know something, it doesn't say "I don't know" — it predicts what a plausible answer *would look like* and generates that. The output sounds confident because that's what training data looks like.

**Three root causes:**
1. **Knowledge cutoff** — the model has no information after its training date
2. **Rare or local facts** — if something appeared rarely in training data, the model's "memory" is weak
3. **No grounding** — the model has no way to check its own answers

**Solutions we'll cover later:**
- RAG (Day 1, Session 5) — give the model your documents at query time
- LLM evaluation (Day 2, Session 2) — detect hallucinations automatically
- Tool use / web search (Day 3+) — let agents look things up

In [ ]:
# The "I don't know" fix — system prompt engineering
# We'll go deep on this in the next notebook, but here's a preview

def ask_with_system(system, user, temperature=0.0):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user}
        ],
        temperature=temperature,
        max_tokens=200
    )
    return response.choices[0].message.content

system_prompt = """You are a helpful assistant. 
If you are not certain about a fact — especially about specific people, 
institutions, or recent events in India — say clearly: 
'I'm not certain about this. Please verify with an authoritative source.'
Never guess or make up facts."""

print(ask_with_system(system_prompt, "Who is the current Vice Chancellor of Anna University?"))

---

## Part 6 — Model Families & What to Use When

In [ ]:
# Quick model cheat-sheet
models = [
    {"name": "gpt-4o-mini",         "provider": "OpenAI",    "type": "Frontier",    "cost": "₹0.01/1K tokens", "best_for": "Everyday tasks, cheap experimentation"},
    {"name": "gpt-4o",              "provider": "OpenAI",    "type": "Frontier",    "cost": "₹1.25/1K tokens","best_for": "Complex reasoning, production"},
    {"name": "claude-haiku-4-5",    "provider": "Anthropic", "type": "Frontier",    "cost": "₹0.08/1K tokens","best_for": "Fast, cheap, good at following instructions"},
    {"name": "claude-sonnet-4-6",   "provider": "Anthropic", "type": "Frontier",    "cost": "₹2.5/1K tokens", "best_for": "Balanced power + cost"},
    {"name": "gemini-2.0-flash",    "provider": "Google",    "type": "Frontier",    "cost": "Free tier",       "best_for": "Multimodal, long context, free to start"},
    {"name": "llama3.2 (Ollama)",   "provider": "Meta",      "type": "Open Source", "cost": "Free (local)",   "best_for": "Privacy-sensitive tasks, no internet needed"},
    {"name": "mistral:7b (Ollama)", "provider": "Mistral",   "type": "Open Source", "cost": "Free (local)",   "best_for": "Lightweight, fast on modest hardware"},
]

print(f"{'Model':<25} {'Provider':<12} {'Type':<13} {'Cost':<20} {'Best For'}")
print("-" * 105)
for m in models:
    print(f"{m['name']:<25} {m['provider']:<12} {m['type']:<13} {m['cost']:<20} {m['best_for']}")

---

## ✅ Key Takeaways

1. **LLMs predict the next token** — they don't retrieve, look up, or reason like a human
2. **Tokens ≠ words** — Indian languages cost more tokens; budget accordingly
3. **Temperature** controls the creativity/consistency tradeoff
4. **Context window** = the model's working memory — bigger is better but more expensive
5. **Hallucination** is a fundamental property, not a bug — mitigate with RAG, evals, and good prompts
6. **Model choice** = cost/capability tradeoff — use the cheapest model that gets the job done

---

## 🎯 Your Turn (Mini Assignment)

Try this before moving to the next notebook:

1. Tokenise the name of your college and the city it's in. How many tokens?
2. Ask GPT-4o-mini a question it will likely hallucinate (specific local fact about your institution)
3. Then add a system prompt that tells it to say "I'm not sure" instead of guessing
4. Compare the outputs

```python
# Your code here
```

---

**Next:** [02 — API Calls: Frontier & Open-Source](./02_api_calls_frontier_opensource.ipynb)